> **Cómputo Paralelo · 6BM1 · ESCOM-IPN · 27/1** · Unidad V · Sesión S04 · lunes 30 de noviembre de 2026 · 10:30–12:00 · Salón 3208

# S04 · Rendimiento en GPU: práctica guiada (PRIMM)

Este cuaderno corre la práctica guiada de la sesión en una GPU de Colab o Kaggle. Antes de ejecutar nada:

1. **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)** en Colab; en Kaggle activa el acelerador GPU.
2. Ten a la mano tu hoja de **predicciones** del `README.md` (fase *Predecir*). No corras las celdas de medición
   sin haberlas escrito.
3. Los tiempos dependen de la GPU que te toque (T4 típica en Colab; P100 o T4 en Kaggle) y del día; anota
   modelo de GPU y fecha (protocolo de medición, §1.4).

Si Kaggle te asigna una **P100**, cambia `-arch=sm_75` por `-arch=sm_60` en la celda de compilación.

## 1. Verificar la GPU asignada

In [ ]:
!nvidia-smi

In [ ]:
# Capacidad de cómputo de la GPU asignada: 7.5 = T4 (sm_75), 6.0 = P100 (sm_60), 8.0 = A100, 8.9 = L4.
import subprocess
cc = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
                    capture_output=True, text=True).stdout.strip()
print(cc or "No se detectó GPU: revisa el tipo de entorno de ejecución")
if cc and float(cc.split(",")[-1]) < 7.5:
    print("AVISO: capacidad < 7.5. Edita la celda de compilación y usa -arch=sm_60 en lugar de sm_75.")

## 2. Escribir los fuentes en el disco de la máquina virtual

In [ ]:
%%writefile medir_eventos.cu
// medir_eventos.cu — Ejemplo resuelto S04: cuatro formas de medir el mismo kernel (saxpy)
//
// Compilar en Colab/Kaggle:  nvcc -O2 -arch=sm_75 -o medir_eventos medir_eventos.cu   (sm_60 en una P100)
// Verificar sin GPU:         nvcc -c -arch=sm_75 medir_eventos.cu
// Uso:                       ./medir_eventos [n]        (por defecto n = 2^24 = 16 777 216)
//
// Protocolo del curso: 1 calentamiento que no cuenta + 5 repeticiones; se reporta la mediana (y mín/máx).
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <ctime>
#include <cuda_runtime.h>

// Revisa el código de error de cada llamada a la API. Sin esto un fallo pasa en silencio (idea errónea 27).
#define CUDA_CHECK(call)                                                          \
    do {                                                                          \
        cudaError_t err__ = (call);                                               \
        if (err__ != cudaSuccess) {                                               \
            fprintf(stderr, "Error CUDA en %s:%d: %s\n", __FILE__, __LINE__,     \
                    cudaGetErrorString(err__));                                   \
            exit(EXIT_FAILURE);                                                   \
        }                                                                         \
    } while (0)

static const int N_REPS = 5;        // repeticiones medidas (impar: la mediana es el elemento central)
static const int BLOCK_SIZE = 256;  // múltiplo de 32; se estudia en bench_gpu.cu

// y[i] = alpha * x[i] + y[i]. Un hilo por elemento; la guarda i < n protege al último bloque.
__global__ void saxpy_kernel(float alpha, const float *x, float *y, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) y[i] = alpha * x[i] + y[i];
}

// Cronómetro de CPU en segundos. Es el equivalente de omp_get_wtime() / MPI_Wtime().
static double cpu_seconds(void)
{
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec + 1e-9 * (double)ts.tv_nsec;
}

static int compare_double(const void *a, const void *b)
{
    double da = *(const double *)a, db = *(const double *)b;
    return (da > db) - (da < db);
}

// Ordena en sitio y devuelve la mediana (N_REPS es impar).
static double median_ms(double *v, int n)
{
    qsort(v, (size_t)n, sizeof(double), compare_double);
    return v[n / 2];
}

static void print_row(const char *label, double *v)
{
    double med = median_ms(v, N_REPS);  // deja v ordenado: v[0] es el mínimo, v[N_REPS-1] el máximo
    printf("%-44s %12.4f %10.4f %10.4f\n", label, med, v[0], v[N_REPS - 1]);
}

int main(int argc, char **argv)
{
    int n = (argc > 1) ? atoi(argv[1]) : (1 << 24);
    if (n <= 0) {
        fprintf(stderr, "n debe ser positivo\n");
        return EXIT_FAILURE;
    }
    size_t bytes = (size_t)n * sizeof(float);
    const float alpha = 2.0f;

    float *h_x = (float *)malloc(bytes);
    float *h_y = (float *)malloc(bytes);
    float *h_out = (float *)malloc(bytes);
    if (!h_x || !h_y || !h_out) {
        fprintf(stderr, "Sin memoria en el anfitrión\n");
        return EXIT_FAILURE;
    }
    for (int i = 0; i < n; ++i) {
        h_x[i] = (float)(i % 1000) * 0.001f;
        h_y[i] = 1.0f;
    }

    // --- Arranque frío: la primera llamada a la API crea el contexto CUDA (decenas o cientos de ms).
    double t0 = cpu_seconds();
    CUDA_CHECK(cudaFree(0));  // llamada "vacía" que fuerza la creación del contexto
    double t_context_ms = (cpu_seconds() - t0) * 1e3;

    float *d_x = NULL, *d_y = NULL;
    CUDA_CHECK(cudaMalloc((void **)&d_x, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_y, bytes));
    CUDA_CHECK(cudaMemcpy(d_x, h_x, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_y, h_y, bytes, cudaMemcpyHostToDevice));

    int grid = (n + BLOCK_SIZE - 1) / BLOCK_SIZE;
    cudaEvent_t ev_start, ev_stop;
    CUDA_CHECK(cudaEventCreate(&ev_start));
    CUDA_CHECK(cudaEventCreate(&ev_stop));

    // --- Calentamiento: el primer lanzamiento carga el módulo y llena cachés; no se cuenta.
    double t_first = cpu_seconds();
    saxpy_kernel<<<grid, BLOCK_SIZE>>>(alpha, d_x, d_y, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());
    double t_first_ms = (cpu_seconds() - t_first) * 1e3;

    double t_nosync[N_REPS], t_sync[N_REPS], t_event[N_REPS], t_total[N_REPS];
    float ms = 0.0f;

    for (int r = 0; r < N_REPS; ++r) {
        // Forma 1 (INCORRECTA): cronómetro de CPU sin sincronizar. El lanzamiento es asíncrono: la CPU
        // sigue antes de que el kernel termine, así que esto mide solo el costo de encolarlo.
        CUDA_CHECK(cudaMemcpy(d_y, h_y, bytes, cudaMemcpyHostToDevice));  // restaura y (saxpy es en sitio)
        double t = cpu_seconds();
        saxpy_kernel<<<grid, BLOCK_SIZE>>>(alpha, d_x, d_y, n);
        t_nosync[r] = (cpu_seconds() - t) * 1e3;
        CUDA_CHECK(cudaDeviceSynchronize());  // fuera del cronómetro: por eso la medición miente

        // Forma 2: cronómetro de CPU + cudaDeviceSynchronize(). Correcta; incluye la espera de la CPU.
        CUDA_CHECK(cudaMemcpy(d_y, h_y, bytes, cudaMemcpyHostToDevice));
        t = cpu_seconds();
        saxpy_kernel<<<grid, BLOCK_SIZE>>>(alpha, d_x, d_y, n);
        CUDA_CHECK(cudaDeviceSynchronize());
        t_sync[r] = (cpu_seconds() - t) * 1e3;

        // Forma 3: eventos en el stream alrededor del kernel. Se mide con el reloj de la GPU.
        CUDA_CHECK(cudaMemcpy(d_y, h_y, bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaEventRecord(ev_start, 0));
        saxpy_kernel<<<grid, BLOCK_SIZE>>>(alpha, d_x, d_y, n);
        CUDA_CHECK(cudaEventRecord(ev_stop, 0));
        CUDA_CHECK(cudaEventSynchronize(ev_stop));  // espera a que el evento de fin se complete
        CUDA_CHECK(cudaEventElapsedTime(&ms, ev_start, ev_stop));
        t_event[r] = ms;

        // Forma 4: eventos alrededor de H2D + kernel + D2H. Es lo que paga quien tiene los datos en la CPU.
        CUDA_CHECK(cudaEventRecord(ev_start, 0));
        CUDA_CHECK(cudaMemcpy(d_x, h_x, bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_y, h_y, bytes, cudaMemcpyHostToDevice));
        saxpy_kernel<<<grid, BLOCK_SIZE>>>(alpha, d_x, d_y, n);
        CUDA_CHECK(cudaMemcpy(h_out, d_y, bytes, cudaMemcpyDeviceToHost));
        CUDA_CHECK(cudaEventRecord(ev_stop, 0));
        CUDA_CHECK(cudaEventSynchronize(ev_stop));
        CUDA_CHECK(cudaEventElapsedTime(&ms, ev_start, ev_stop));
        t_total[r] = ms;
    }
    CUDA_CHECK(cudaGetLastError());

    // --- Verificación contra la CPU: siempre, antes de creer cualquier tiempo.
    double max_err = 0.0;
    for (int i = 0; i < n; ++i) {
        double ref = (double)alpha * h_x[i] + h_y[i];
        double err = fabs(ref - h_out[i]);
        if (err > max_err) max_err = err;
    }

    printf("medir_eventos: saxpy con n = %d (%.1f MB por vector), bloque = %d, malla = %d\n",
           n, (double)bytes / 1048576.0, BLOCK_SIZE, grid);
    printf("Primera llamada CUDA (creación del contexto): %.1f ms\n", t_context_ms);
    printf("Primer lanzamiento del kernel (frío):          %.3f ms  <- por eso hay calentamiento\n", t_first_ms);
    printf("\n%-44s %12s %10s %10s\n", "Forma de medir", "mediana ms", "min", "max");
    print_row("1. CPU sin sincronizar (INCORRECTA)", t_nosync);
    print_row("2. CPU + cudaDeviceSynchronize", t_sync);
    print_row("3. cudaEvent alrededor del kernel", t_event);
    print_row("4. cudaEvent H2D + kernel + D2H", t_total);
    double med_kernel = t_event[N_REPS / 2], med_total = t_total[N_REPS / 2];  // ya ordenados
    printf("\nCopias = %.1f %% del total (total %.3f ms, kernel %.3f ms)\n",
           100.0 * (med_total - med_kernel) / med_total, med_total, med_kernel);
    printf("Ancho de banda efectivo del kernel: %.1f GB/s (%zu bytes leídos+escritos)\n",
           3.0 * (double)bytes / (med_kernel * 1e-3) / 1e9, 3 * bytes);
    printf("Verificación contra CPU: %s (error máximo %.2e)\n", max_err < 1e-5 ? "OK" : "FALLA", max_err);

    CUDA_CHECK(cudaEventDestroy(ev_start));
    CUDA_CHECK(cudaEventDestroy(ev_stop));
    CUDA_CHECK(cudaFree(d_x));
    CUDA_CHECK(cudaFree(d_y));
    free(h_x);
    free(h_y);
    free(h_out);
    CUDA_CHECK(cudaDeviceReset());  // vacía los búferes del perfilador (nsys) antes de salir
    return (max_err < 1e-5) ? EXIT_SUCCESS : EXIT_FAILURE;
}

In [ ]:
%%writefile bench_gpu.cu
// bench_gpu.cu — Práctica guiada S04: barrido de tamaño de bloque y de tamaño del problema, con los
// tiempos de copia (H2D, D2H) y de kernel medidos por separado con cudaEvent_t.
//
// Compilar en Colab/Kaggle:  nvcc -O2 -arch=sm_75 -o bench_gpu bench_gpu.cu   (sm_60 en una P100)
// Verificar sin GPU:         nvcc -c -arch=sm_75 bench_gpu.cu
//
// Uso:
//   ./bench_gpu bloque <n>      <salida.csv> [pinned]   barrido de bloque {32,64,...,1024} con n fijo
//   ./bench_gpu umbral <bloque> <salida.csv> [pinned]   barrido de n = 2^10, 2^12, ..., 2^26 con bloque fijo
//
// Imprime una tabla y AGREGA filas al CSV (escribe el encabezado si el archivo está vacío), así varias
// corridas se juntan en un solo archivo que lee graficar.py.
// Protocolo del curso: 1 calentamiento + 5 repeticiones; se reporta la mediana. Todo se verifica contra la CPU.
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <cmath>
#include <ctime>
#include <cuda_runtime.h>

#define CUDA_CHECK(call)                                                          \
    do {                                                                          \
        cudaError_t err__ = (call);                                               \
        if (err__ != cudaSuccess) {                                               \
            fprintf(stderr, "Error CUDA en %s:%d: %s\n", __FILE__, __LINE__,     \
                    cudaGetErrorString(err__));                                   \
            exit(EXIT_FAILURE);                                                   \
        }                                                                         \
    } while (0)

static const int N_REPS = 5;
static const int BLOCK_SIZES[] = {32, 64, 128, 256, 512, 1024};  // Modificar: agrega 48 o 96 y predice antes
static const int N_BLOCK_SIZES = (int)(sizeof(BLOCK_SIZES) / sizeof(BLOCK_SIZES[0]));
static const int LOG2_N_MIN = 10, LOG2_N_MAX = 26, LOG2_N_STEP = 2;
static const float ALPHA = 2.0f;

typedef enum { K_VECTOR_ADD = 0, K_SAXPY = 1 } KernelId;
static const char *KERNEL_NAMES[] = {"vector_add", "saxpy"};

typedef struct {
    double h2d_ms, kernel_ms, d2h_ms;  // medianas de N_REPS repeticiones
} GpuTimes;

// ----- Kernels: un hilo por elemento, dos vectores de entrada y uno de salida (12 bytes por elemento).
__global__ void vector_add_kernel(const float *a, const float *b, float *c, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];  // 1 FLOP por 12 bytes: intensidad aritmética 0.083 FLOP/byte
}

__global__ void saxpy_kernel(float alpha, const float *x, const float *y, float *out, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) out[i] = alpha * x[i] + y[i];  // 2 FLOP por 12 bytes: 0.167 FLOP/byte
}

// ----- Referencias en CPU (el "mejor programa secuencial" contra el que se calcula el speedup).
static void vector_add_cpu(const float *a, const float *b, float *c, int n)
{
    for (int i = 0; i < n; ++i) c[i] = a[i] + b[i];
}

static void saxpy_cpu(float alpha, const float *x, const float *y, float *out, int n)
{
    for (int i = 0; i < n; ++i) out[i] = alpha * x[i] + y[i];
}

// ----- Utilidades de medición.
static double cpu_seconds(void)
{
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec + 1e-9 * (double)ts.tv_nsec;
}

static int compare_double(const void *a, const void *b)
{
    double da = *(const double *)a, db = *(const double *)b;
    return (da > db) - (da < db);
}

static double median_of(double *v, int n)
{
    qsort(v, (size_t)n, sizeof(double), compare_double);
    return v[n / 2];
}

static void *host_alloc(size_t bytes, int pinned)
{
    void *p = NULL;
    if (pinned) {
        CUDA_CHECK(cudaMallocHost(&p, bytes));  // memoria fija (pinned): la copia va por DMA sin paso intermedio
    } else {
        p = malloc(bytes);  // memoria paginable: el driver copia primero a un búfer fijo interno
        if (!p) {
            fprintf(stderr, "Sin memoria en el anfitrión (%zu bytes)\n", bytes);
            exit(EXIT_FAILURE);
        }
    }
    return p;
}

static void host_free(void *p, int pinned)
{
    if (pinned) CUDA_CHECK(cudaFreeHost(p));
    else free(p);
}

static void init_inputs(float *a, float *b, int n)
{
    for (int i = 0; i < n; ++i) {
        a[i] = (float)(i % 1000) * 0.001f;
        b[i] = 1.0f + (float)(i % 7);
    }
}

static double max_abs_error(const float *got, const float *ref, int n)
{
    double max_err = 0.0;
    for (int i = 0; i < n; ++i) {
        double err = fabs((double)got[i] - (double)ref[i]);
        if (err > max_err) max_err = err;
    }
    return max_err;
}

// Mediana del tiempo de la versión secuencial (1 calentamiento + N_REPS repeticiones). Deja el resultado en c.
static double measure_cpu_ms(KernelId k, int n, const float *a, const float *b, float *c)
{
    double t[N_REPS];
    for (int r = -1; r < N_REPS; ++r) {
        double t0 = cpu_seconds();
        if (k == K_VECTOR_ADD) vector_add_cpu(a, b, c, n);
        else saxpy_cpu(ALPHA, a, b, c, n);
        double dt = (cpu_seconds() - t0) * 1e3;
        if (r >= 0) t[r] = dt;
    }
    return median_of(t, N_REPS);
}

// Mide H2D, kernel y D2H por separado con cuatro eventos en el stream 0. Deja la salida de la última
// repetición en h_c para verificarla.
static GpuTimes measure_gpu(KernelId k, int n, int block, const float *h_a, const float *h_b, float *h_c)
{
    size_t bytes = (size_t)n * sizeof(float);
    float *d_a = NULL, *d_b = NULL, *d_c = NULL;
    CUDA_CHECK(cudaMalloc((void **)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_b, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_c, bytes));

    cudaEvent_t ev[4];
    for (int i = 0; i < 4; ++i) CUDA_CHECK(cudaEventCreate(&ev[i]));

    int grid = (n + block - 1) / block;
    double h2d[N_REPS], ker[N_REPS], d2h[N_REPS];

    for (int r = -1; r < N_REPS; ++r) {  // r = -1 es el calentamiento
        CUDA_CHECK(cudaEventRecord(ev[0], 0));
        CUDA_CHECK(cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaEventRecord(ev[1], 0));
        if (k == K_VECTOR_ADD) vector_add_kernel<<<grid, block>>>(d_a, d_b, d_c, n);
        else saxpy_kernel<<<grid, block>>>(ALPHA, d_a, d_b, d_c, n);
        CUDA_CHECK(cudaEventRecord(ev[2], 0));
        CUDA_CHECK(cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost));
        CUDA_CHECK(cudaEventRecord(ev[3], 0));
        CUDA_CHECK(cudaEventSynchronize(ev[3]));
        CUDA_CHECK(cudaGetLastError());  // un bloque de 1024 con demasiados registros falla aquí, no antes

        float ms;
        if (r >= 0) {
            CUDA_CHECK(cudaEventElapsedTime(&ms, ev[0], ev[1]));
            h2d[r] = ms;
            CUDA_CHECK(cudaEventElapsedTime(&ms, ev[1], ev[2]));
            ker[r] = ms;
            CUDA_CHECK(cudaEventElapsedTime(&ms, ev[2], ev[3]));
            d2h[r] = ms;
        }
    }

    GpuTimes t;
    t.h2d_ms = median_of(h2d, N_REPS);
    t.kernel_ms = median_of(ker, N_REPS);
    t.d2h_ms = median_of(d2h, N_REPS);

    for (int i = 0; i < 4; ++i) CUDA_CHECK(cudaEventDestroy(ev[i]));
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    return t;
}

// Ocupación teórica: warps residentes por SM que permite este kernel con este bloque, entre el máximo del SM.
static double occupancy_pct(KernelId k, int block, int max_threads_per_sm)
{
    int blocks_per_sm = 0;
    if (k == K_VECTOR_ADD)
        CUDA_CHECK(cudaOccupancyMaxActiveBlocksPerMultiprocessor(&blocks_per_sm, vector_add_kernel, block, 0));
    else
        CUDA_CHECK(cudaOccupancyMaxActiveBlocksPerMultiprocessor(&blocks_per_sm, saxpy_kernel, block, 0));
    int warps_per_block = (block + 31) / 32;  // un bloque de 48 hilos ocupa 2 warps, uno de ellos a medias
    int max_warps = max_threads_per_sm / 32;
    return 100.0 * blocks_per_sm * warps_per_block / max_warps;
}

// ----- Salida: tabla en pantalla y fila CSV.
static const char *CSV_HEADER =
    "experimento,kernel,n,block,grid,ocupacion_pct,memoria,t_h2d_ms,t_kernel_ms,t_d2h_ms,t_total_ms,"
    "t_cpu_ms,speedup_kernel,speedup_total,ancho_banda_gbs,origen,dispositivo\n";

static FILE *open_csv(const char *path)
{
    FILE *f = fopen(path, "a");
    if (!f) {
        fprintf(stderr, "No se pudo abrir %s\n", path);
        exit(EXIT_FAILURE);
    }
    fseek(f, 0, SEEK_END);
    if (ftell(f) == 0) fputs(CSV_HEADER, f);
    return f;
}

static void print_table_header(void)
{
    printf("%-10s %10s %6s %8s %6s %9s %10s %9s %10s %10s %8s %8s %7s\n", "kernel", "n", "bloque",
           "malla", "ocup%", "H2D ms", "kernel ms", "D2H ms", "total ms", "CPU ms", "S_kern", "S_total",
           "GB/s");
}

static void emit_row(FILE *csv, const char *experiment, KernelId k, int n, int block, double occ,
                     const char *memory, GpuTimes t, double cpu_ms, const char *device)
{
    int grid = (n + block - 1) / block;
    double total = t.h2d_ms + t.kernel_ms + t.d2h_ms;
    double bytes_moved = 3.0 * (double)n * sizeof(float);  // 2 lecturas + 1 escritura por elemento
    double gbs = bytes_moved / (t.kernel_ms * 1e-3) / 1e9;
    double s_kernel = cpu_ms / t.kernel_ms, s_total = cpu_ms / total;
    printf("%-10s %10d %6d %8d %6.0f %9.4f %10.4f %9.4f %10.4f %10.4f %8.2f %8.2f %7.1f\n",
           KERNEL_NAMES[k], n, block, grid, occ, t.h2d_ms, t.kernel_ms, t.d2h_ms, total, cpu_ms, s_kernel,
           s_total, gbs);
    fprintf(csv, "%s,%s,%d,%d,%d,%.1f,%s,%.6f,%.6f,%.6f,%.6f,%.6f,%.4f,%.4f,%.2f,gpu,%s\n", experiment,
            KERNEL_NAMES[k], n, block, grid, occ, memory, t.h2d_ms, t.kernel_ms, t.d2h_ms, total, cpu_ms,
            s_kernel, s_total, gbs, device);
}

static void usage(const char *prog)
{
    fprintf(stderr,
            "Uso:\n  %s bloque <n>      <salida.csv> [pinned]\n  %s umbral <bloque> <salida.csv> [pinned]\n",
            prog, prog);
    exit(EXIT_FAILURE);
}

int main(int argc, char **argv)
{
    if (argc < 4) usage(argv[0]);
    const char *mode = argv[1];
    long param = atol(argv[2]);
    const char *csv_path = argv[3];
    int pinned = (argc > 4 && strcmp(argv[4], "pinned") == 0);
    const char *memory = pinned ? "pinned" : "pageable";
    int is_block_sweep = (strcmp(mode, "bloque") == 0);
    if (!is_block_sweep && strcmp(mode, "umbral") != 0) usage(argv[0]);

    // --- Entorno (protocolo §1.4): se anota el dispositivo y su ancho de banda teórico.
    int dev = 0, mem_clock_khz = 0, bus_width_bits = 0;
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDevice(&dev));
    CUDA_CHECK(cudaGetDeviceProperties(&prop, dev));
    CUDA_CHECK(cudaDeviceGetAttribute(&mem_clock_khz, cudaDevAttrMemoryClockRate, dev));
    CUDA_CHECK(cudaDeviceGetAttribute(&bus_width_bits, cudaDevAttrGlobalMemoryBusWidth, dev));
    double peak_gbs = 2.0 * mem_clock_khz * 1e3 * (bus_width_bits / 8.0) / 1e9;  // DDR: 2 transferencias/ciclo
    char device[256];  // prop.name mide 256 bytes
    snprintf(device, sizeof device, "%s", prop.name);
    for (char *p = device; *p; ++p)
        if (*p == ',') *p = ' ';  // el nombre va a un CSV
    printf("GPU: %s · %d SM · %d hilos/SM · ancho de banda teórico %.0f GB/s · memoria del anfitrión: %s\n",
           device, prop.multiProcessorCount, prop.maxThreadsPerMultiProcessor, peak_gbs, memory);

    FILE *csv = open_csv(csv_path);
    int failures = 0;

    if (is_block_sweep) {
        int n = (int)param;
        if (n <= 0 || n > (1 << 27)) {
            fprintf(stderr, "n fuera de rango (1 .. 2^27)\n");
            return EXIT_FAILURE;
        }
        size_t bytes = (size_t)n * sizeof(float);
        float *h_a = (float *)host_alloc(bytes, pinned), *h_b = (float *)host_alloc(bytes, pinned);
        float *h_c = (float *)host_alloc(bytes, pinned), *h_ref = (float *)malloc(bytes);
        if (!h_ref) return EXIT_FAILURE;
        init_inputs(h_a, h_b, n);
        printf("Experimento 'bloque': n = %d (%.1f MB por vector), bloques de %d a %d\n", n,
               (double)bytes / 1048576.0, BLOCK_SIZES[0], BLOCK_SIZES[N_BLOCK_SIZES - 1]);
        print_table_header();
        for (int k = 0; k < 2; ++k) {
            double cpu_ms = measure_cpu_ms((KernelId)k, n, h_a, h_b, h_ref);
            for (int i = 0; i < N_BLOCK_SIZES; ++i) {
                int block = BLOCK_SIZES[i];
                GpuTimes t = measure_gpu((KernelId)k, n, block, h_a, h_b, h_c);
                double err = max_abs_error(h_c, h_ref, n);
                if (err > 1e-5) {
                    printf("FALLA de verificación: %s bloque %d, error %.2e\n", KERNEL_NAMES[k], block, err);
                    ++failures;
                }
                emit_row(csv, "bloque", (KernelId)k, n, block,
                         occupancy_pct((KernelId)k, block, prop.maxThreadsPerMultiProcessor), memory, t,
                         cpu_ms, device);
            }
        }
        host_free(h_a, pinned);
        host_free(h_b, pinned);
        host_free(h_c, pinned);
        free(h_ref);
    } else {
        int block = (int)param;
        if (block < 1 || block > 1024) {
            fprintf(stderr, "El bloque debe estar entre 1 y 1024\n");
            return EXIT_FAILURE;
        }
        if (block % 32 != 0) printf("Aviso: %d no es múltiplo de 32; el último warp de cada bloque va incompleto\n", block);
        printf("Experimento 'umbral': bloque = %d, n = 2^%d .. 2^%d\n", block, LOG2_N_MIN, LOG2_N_MAX);
        print_table_header();
        for (int log2n = LOG2_N_MIN; log2n <= LOG2_N_MAX; log2n += LOG2_N_STEP) {
            int n = 1 << log2n;
            size_t bytes = (size_t)n * sizeof(float);
            float *h_a = (float *)host_alloc(bytes, pinned), *h_b = (float *)host_alloc(bytes, pinned);
            float *h_c = (float *)host_alloc(bytes, pinned), *h_ref = (float *)malloc(bytes);
            if (!h_ref) return EXIT_FAILURE;
            init_inputs(h_a, h_b, n);
            for (int k = 0; k < 2; ++k) {
                double cpu_ms = measure_cpu_ms((KernelId)k, n, h_a, h_b, h_ref);
                GpuTimes t = measure_gpu((KernelId)k, n, block, h_a, h_b, h_c);
                double err = max_abs_error(h_c, h_ref, n);
                if (err > 1e-5) {
                    printf("FALLA de verificación: %s n %d, error %.2e\n", KERNEL_NAMES[k], n, err);
                    ++failures;
                }
                emit_row(csv, "umbral", (KernelId)k, n, block,
                         occupancy_pct((KernelId)k, block, prop.maxThreadsPerMultiProcessor), memory, t,
                         cpu_ms, device);
            }
            host_free(h_a, pinned);
            host_free(h_b, pinned);
            host_free(h_c, pinned);
            free(h_ref);
        }
    }

    fclose(csv);
    printf("Verificación contra CPU: %s. Filas agregadas a %s\n", failures ? "FALLA" : "OK", csv_path);
    CUDA_CHECK(cudaDeviceReset());
    return failures ? EXIT_FAILURE : EXIT_SUCCESS;
}

In [ ]:
%%writefile pinned_streams.cu
// pinned_streams.cu — Panorama S04: memoria paginable vs. pinned vs. pinned + streams (solapar copia y cómputo)
//
// Compilar en Colab/Kaggle:  nvcc -O2 -arch=sm_75 -o pinned_streams pinned_streams.cu   (sm_60 en una P100)
// Verificar sin GPU:         nvcc -c -arch=sm_75 pinned_streams.cu
// Uso:                       ./pinned_streams [n] [n_streams]      (por defecto n = 2^24, 4 streams)
//
// Tres maneras de calcular out = alpha*x + y cuando los datos empiezan y terminan en la CPU:
//   A. malloc (memoria paginable) + cudaMemcpy síncrono        H2D -> kernel -> D2H, todo en serie
//   B. cudaMallocHost (memoria pinned) + cudaMemcpy síncrono   la misma serie, pero la copia va por DMA directo
//   C. pinned + cudaMemcpyAsync en varios streams              el trozo k se copia mientras el trozo k-1 se calcula
// Protocolo: 1 calentamiento + 5 repeticiones; mediana. Cada versión se verifica contra la CPU.
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

#define CUDA_CHECK(call)                                                          \
    do {                                                                          \
        cudaError_t err__ = (call);                                               \
        if (err__ != cudaSuccess) {                                               \
            fprintf(stderr, "Error CUDA en %s:%d: %s\n", __FILE__, __LINE__,     \
                    cudaGetErrorString(err__));                                   \
            exit(EXIT_FAILURE);                                                   \
        }                                                                         \
    } while (0)

static const int N_REPS = 5;
static const int BLOCK_SIZE = 256;
static const int MAX_STREAMS = 16;

__global__ void saxpy_kernel(float alpha, const float *x, const float *y, float *out, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) out[i] = alpha * x[i] + y[i];
}

static int compare_double(const void *a, const void *b)
{
    double da = *(const double *)a, db = *(const double *)b;
    return (da > db) - (da < db);
}

typedef struct {
    int n, n_streams;
    float alpha;
    const float *h_x, *h_y;  // entradas (paginables o pinned según la versión)
    float *h_out;            // salida del anfitrión
    float *d_x, *d_y, *d_out;
    cudaStream_t *streams;
} Config;

// Versión A o B: tres operaciones síncronas en el stream 0. Lo único que cambia es de dónde viene h_x/h_y.
static void run_sync(const Config *c)
{
    size_t bytes = (size_t)c->n * sizeof(float);
    int grid = (c->n + BLOCK_SIZE - 1) / BLOCK_SIZE;
    CUDA_CHECK(cudaMemcpy(c->d_x, c->h_x, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(c->d_y, c->h_y, bytes, cudaMemcpyHostToDevice));
    saxpy_kernel<<<grid, BLOCK_SIZE>>>(c->alpha, c->d_x, c->d_y, c->d_out, c->n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(c->h_out, c->d_out, bytes, cudaMemcpyDeviceToHost));
}

// Versión C: el vector se parte en n_streams trozos; cada trozo tiene su propia secuencia H2D -> kernel -> D2H
// en su stream. Los motores de copia y los SM trabajan en paralelo sobre trozos distintos.
// Requisito: memoria pinned. Con memoria paginable cudaMemcpyAsync se vuelve síncrona y no hay solape.
static void run_streams(const Config *c)
{
    int chunk = (c->n + c->n_streams - 1) / c->n_streams;
    for (int s = 0; s < c->n_streams; ++s) {
        int offset = s * chunk;
        int len = c->n - offset;
        if (len <= 0) break;
        if (len > chunk) len = chunk;
        size_t bytes = (size_t)len * sizeof(float);
        int grid = (len + BLOCK_SIZE - 1) / BLOCK_SIZE;
        cudaStream_t st = c->streams[s];
        CUDA_CHECK(cudaMemcpyAsync(c->d_x + offset, c->h_x + offset, bytes, cudaMemcpyHostToDevice, st));
        CUDA_CHECK(cudaMemcpyAsync(c->d_y + offset, c->h_y + offset, bytes, cudaMemcpyHostToDevice, st));
        saxpy_kernel<<<grid, BLOCK_SIZE, 0, st>>>(c->alpha, c->d_x + offset, c->d_y + offset,
                                                  c->d_out + offset, len);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaMemcpyAsync(c->h_out + offset, c->d_out + offset, bytes, cudaMemcpyDeviceToHost, st));
    }
    // No hace falta cudaStreamSynchronize aquí: el evento de fin se graba en el stream 0 (legado), que espera
    // a todos los streams "bloqueantes" creados con cudaStreamCreate.
}

// Mide una versión completa (copias + kernel) con eventos en el stream 0; devuelve la mediana en ms.
static double measure_ms(const Config *c, int use_streams)
{
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    double t[N_REPS];
    for (int r = -1; r < N_REPS; ++r) {
        CUDA_CHECK(cudaEventRecord(start, 0));
        if (use_streams) run_streams(c);
        else run_sync(c);
        CUDA_CHECK(cudaEventRecord(stop, 0));
        CUDA_CHECK(cudaEventSynchronize(stop));
        float ms;
        CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
        if (r >= 0) t[r] = ms;
    }
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    qsort(t, N_REPS, sizeof(double), compare_double);
    printf("   (min %.3f ms, max %.3f ms)\n", t[0], t[N_REPS - 1]);
    return t[N_REPS / 2];
}

static int verify(const float *out, const float *x, const float *y, float alpha, int n)
{
    double max_err = 0.0;
    for (int i = 0; i < n; ++i) {
        double err = fabs((double)alpha * x[i] + y[i] - out[i]);
        if (err > max_err) max_err = err;
    }
    return max_err < 1e-5;
}

int main(int argc, char **argv)
{
    int n = (argc > 1) ? atoi(argv[1]) : (1 << 24);
    int n_streams = (argc > 2) ? atoi(argv[2]) : 4;
    if (n <= 0 || n_streams < 1 || n_streams > MAX_STREAMS) {
        fprintf(stderr, "Uso: %s [n > 0] [1 <= n_streams <= %d]\n", argv[0], MAX_STREAMS);
        return EXIT_FAILURE;
    }
    size_t bytes = (size_t)n * sizeof(float);
    const float alpha = 2.0f;

    // Dos juegos de arreglos del anfitrión: paginables (malloc) y fijos (cudaMallocHost).
    float *pg_x = (float *)malloc(bytes), *pg_y = (float *)malloc(bytes), *pg_out = (float *)malloc(bytes);
    float *pn_x = NULL, *pn_y = NULL, *pn_out = NULL;
    if (!pg_x || !pg_y || !pg_out) {
        fprintf(stderr, "Sin memoria en el anfitrión\n");
        return EXIT_FAILURE;
    }
    CUDA_CHECK(cudaMallocHost((void **)&pn_x, bytes));
    CUDA_CHECK(cudaMallocHost((void **)&pn_y, bytes));
    CUDA_CHECK(cudaMallocHost((void **)&pn_out, bytes));
    for (int i = 0; i < n; ++i) {
        pg_x[i] = pn_x[i] = (float)(i % 1000) * 0.001f;
        pg_y[i] = pn_y[i] = 1.0f + (float)(i % 7);
    }

    float *d_x = NULL, *d_y = NULL, *d_out = NULL;
    CUDA_CHECK(cudaMalloc((void **)&d_x, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_y, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_out, bytes));

    cudaStream_t streams[MAX_STREAMS];
    for (int s = 0; s < n_streams; ++s) CUDA_CHECK(cudaStreamCreate(&streams[s]));

    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("pinned_streams: GPU %s · n = %d (%.1f MB por vector) · %d streams · motores de copia: %d\n", prop.name,
           n, (double)bytes / 1048576.0, n_streams, prop.asyncEngineCount);

    Config a = {n, n_streams, alpha, pg_x, pg_y, pg_out, d_x, d_y, d_out, streams};
    Config b = {n, n_streams, alpha, pn_x, pn_y, pn_out, d_x, d_y, d_out, streams};

    printf("A. paginable, síncrona\n");
    double t_a = measure_ms(&a, 0);
    int ok_a = verify(pg_out, pg_x, pg_y, alpha, n);
    printf("B. pinned (cudaMallocHost), síncrona\n");
    double t_b = measure_ms(&b, 0);
    int ok_b = verify(pn_out, pn_x, pn_y, alpha, n);
    printf("C. pinned + %d streams, asíncrona\n", n_streams);
    double t_c = measure_ms(&b, 1);
    int ok_c = verify(pn_out, pn_x, pn_y, alpha, n);

    printf("\n%-40s %12s %12s\n", "Versión (H2D + kernel + D2H)", "mediana ms", "vs. A");
    printf("%-40s %12.3f %11.2fx\n", "A. paginable, síncrona", t_a, 1.0);
    printf("%-40s %12.3f %11.2fx\n", "B. pinned, síncrona", t_b, t_a / t_b);
    printf("%-40s %12.3f %11.2fx\n", "C. pinned + streams, asíncrona", t_c, t_a / t_c);
    printf("Verificación contra CPU: A %s · B %s · C %s\n", ok_a ? "OK" : "FALLA", ok_b ? "OK" : "FALLA",
           ok_c ? "OK" : "FALLA");

    for (int s = 0; s < n_streams; ++s) CUDA_CHECK(cudaStreamDestroy(streams[s]));
    CUDA_CHECK(cudaFree(d_x));
    CUDA_CHECK(cudaFree(d_y));
    CUDA_CHECK(cudaFree(d_out));
    CUDA_CHECK(cudaFreeHost(pn_x));
    CUDA_CHECK(cudaFreeHost(pn_y));
    CUDA_CHECK(cudaFreeHost(pn_out));
    free(pg_x);
    free(pg_y);
    free(pg_out);
    CUDA_CHECK(cudaDeviceReset());
    return (ok_a && ok_b && ok_c) ? EXIT_SUCCESS : EXIT_FAILURE;
}

In [ ]:
%%writefile graficar.py
"""graficar.py — Grafica los resultados de bench_gpu.cu (o el CSV de ejemplo de bench_cpu --csv-ejemplo).

Uso: python3 graficar.py resultados.csv [carpeta_salida]

Produce en carpeta_salida (por defecto la del CSV):
  tiempo_vs_bloque.png   tiempo del kernel vs. tamaño de bloque (experimento "bloque"); un panel por n
  speedup_vs_n.png       speedup vs. n (experimento "umbral"): solo kernel (línea continua) y con copias
                         (discontinua = paginable, punteada = pinned); la línea S = 1 marca el cruce
y escribe en pantalla la tabla del experimento "umbral" y un resumen: mejor bloque por kernel y primer n a
partir del cual la GPU gana con y sin copias. Si el CSV viene de un modelo (origen=modelo) lo dice en el título.
Solo usa la biblioteca estándar + matplotlib.
"""
import csv
import os
import sys

import matplotlib

matplotlib.use("Agg")  # siempre a archivo: funciona sin pantalla y dentro de Colab
import matplotlib.pyplot as plt  # noqa: E402

COLORS = {"vector_add": "#2a78d6", "saxpy": "#eb6834"}  # paleta validada para daltonismo (2 series)
MARKERS = {"vector_add": ("o", 9), "saxpy": ("s", 5)}  # círculo grande y cuadrado chico
STYLE_TOTAL = {"pageable": "--", "pinned": ":"}
BLOCK_ORDER = (32, 64, 128, 256, 512, 1024)


def read_rows(path):
    with open(path, newline="") as f:
        rows = list(csv.DictReader(f))
    for r in rows:
        for key in ("n", "block", "grid"):
            r[key] = int(r[key])
        for key in ("t_h2d_ms", "t_kernel_ms", "t_d2h_ms", "t_total_ms", "t_cpu_ms", "speedup_kernel",
                    "speedup_total", "ancho_banda_gbs", "ocupacion_pct"):
            r[key] = float(r[key])
    return rows


def source_note(rows):
    origins = {r["origen"] for r in rows}
    if origins == {"gpu"}:
        devices = sorted({r["dispositivo"] for r in rows})
        return "medido en " + ", ".join(devices)
    return "DATOS DE MODELO (sin GPU), no medidos"


def plot_block_sweep(rows, out_path, note):
    rows = [r for r in rows if r["experimento"] == "bloque"]
    if not rows:
        print("Sin filas del experimento 'bloque': no se genera tiempo_vs_bloque.png")
        return
    sizes = sorted({r["n"] for r in rows}, reverse=True)
    fig, axes = plt.subplots(1, len(sizes), figsize=(5.2 * len(sizes), 4.2), squeeze=False)
    for ax, n in zip(axes[0], sizes):
        for kernel in ("vector_add", "saxpy"):
            pts = sorted((r["block"], r["t_kernel_ms"]) for r in rows if r["n"] == n and r["kernel"] == kernel)
            if not pts:
                continue
            xs = [BLOCK_ORDER.index(b) if b in BLOCK_ORDER else -1 for b, _ in pts]
            # Marcadores distintos por kernel: si los tiempos coinciden, las dos series siguen siendo visibles.
            ax.plot(xs, [t for _, t in pts], marker=MARKERS[kernel][0], markersize=MARKERS[kernel][1],
                    linewidth=2, color=COLORS[kernel], label=kernel)
            best = min(pts, key=lambda p: p[1])
            ax.annotate(f"mejor {kernel}: {best[0]}",
                        (BLOCK_ORDER.index(best[0]) if best[0] in BLOCK_ORDER else -1, best[1]),
                        textcoords="offset points", xytext=(0, -14 if kernel == "vector_add" else 10),
                        ha="center", fontsize=8, color="#52514e")
        ax.set_xticks(range(len(BLOCK_ORDER)))
        ax.set_xticklabels([str(b) for b in BLOCK_ORDER])
        ax.set_xlabel("hilos por bloque")
        ax.set_ylabel("tiempo del kernel (ms)")
        ax.set_ylim(bottom=0)
        ax.set_title(f"n = {n:,}".replace(",", " "), fontsize=10)
        ax.grid(axis="y", color="#e5e4df", linewidth=0.8)
        ax.spines[["top", "right"]].set_visible(False)
        ax.legend(frameon=False)
    fig.suptitle(f"Tiempo del kernel vs. tamaño de bloque · {note}", fontsize=11)
    fig.tight_layout()
    fig.savefig(out_path, dpi=130)
    plt.close(fig)
    print(f"Gráfica escrita: {out_path}")


def plot_threshold(rows, out_path, note):
    rows = [r for r in rows if r["experimento"] == "umbral"]
    if not rows:
        print("Sin filas del experimento 'umbral': no se genera speedup_vs_n.png")
        return
    memories = [m for m in ("pageable", "pinned") if any(r["memoria"] == m for r in rows)]
    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    for kernel in ("vector_add", "saxpy"):
        base = [r for r in rows if r["kernel"] == kernel and r["memoria"] == memories[0]]
        base.sort(key=lambda r: r["n"])
        if base:
            ax.plot([r["n"] for r in base], [r["speedup_kernel"] for r in base], "-", marker="o", markersize=6,
                    linewidth=2, color=COLORS[kernel], label=f"{kernel}: solo kernel")
        for mem in memories:
            sub = sorted((r for r in rows if r["kernel"] == kernel and r["memoria"] == mem), key=lambda r: r["n"])
            if sub:
                ax.plot([r["n"] for r in sub], [r["speedup_total"] for r in sub], STYLE_TOTAL[mem], marker="s",
                        markersize=5, linewidth=2, color=COLORS[kernel], label=f"{kernel}: con copias ({mem})")
    ax.axhline(1.0, color="#52514e", linewidth=1, linestyle="-")
    ax.text(min(r["n"] for r in rows), 1.0, " S = 1 (empate con la CPU)", va="bottom", fontsize=8, color="#52514e")
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xlabel("n (elementos por vector)")
    ax.set_ylabel("speedup = T_cpu / T_gpu")
    ax.set_title(f"¿A partir de qué n gana la GPU? · {note}", fontsize=11)
    ax.grid(color="#e5e4df", linewidth=0.8, which="major")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(out_path, dpi=130)
    plt.close(fig)
    print(f"Gráfica escrita: {out_path}")


def first_crossing(sub, key):
    """Primer n (ordenado) con speedup > 1, o None."""
    for r in sorted(sub, key=lambda r: r["n"]):
        if r[key] > 1.0:
            return r["n"]
    return None


def print_summary(rows):
    block_rows = [r for r in rows if r["experimento"] == "bloque"]
    if block_rows:
        print("\nResumen 'bloque' (tiempo del kernel):")
        for n in sorted({r["n"] for r in block_rows}, reverse=True):
            for kernel in ("vector_add", "saxpy"):
                sub = [r for r in block_rows if r["n"] == n and r["kernel"] == kernel]
                if not sub:
                    continue
                best = min(sub, key=lambda r: r["t_kernel_ms"])
                worst = max(sub, key=lambda r: r["t_kernel_ms"])
                print(f"  n={n:>10} {kernel:<11} mejor bloque {best['block']:>4} ({best['t_kernel_ms']:.4f} ms), "
                      f"peor {worst['block']:>4} ({worst['t_kernel_ms']:.4f} ms), "
                      f"peor/mejor = {worst['t_kernel_ms'] / best['t_kernel_ms']:.2f}x")
    thr_rows = [r for r in rows if r["experimento"] == "umbral"]
    if thr_rows:
        print("\nTabla 'umbral' (ms; mediana de 5):")
        print("| kernel | memoria | n | H2D | kernel | D2H | total | CPU | S_kernel | S_total | copias % |")
        print("|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|")
        for r in sorted(thr_rows, key=lambda r: (r["kernel"], r["memoria"], r["n"])):
            copies = 100.0 * (r["t_h2d_ms"] + r["t_d2h_ms"]) / r["t_total_ms"]
            print(f"| {r['kernel']} | {r['memoria']} | {r['n']} | {r['t_h2d_ms']:.4f} | {r['t_kernel_ms']:.4f} | "
                  f"{r['t_d2h_ms']:.4f} | {r['t_total_ms']:.4f} | {r['t_cpu_ms']:.4f} | {r['speedup_kernel']:.2f} | "
                  f"{r['speedup_total']:.2f} | {copies:.0f} |")
        print("\nResumen 'umbral' (primer n con speedup > 1):")
        for kernel in ("vector_add", "saxpy"):
            for mem in ("pageable", "pinned"):
                sub = [r for r in thr_rows if r["kernel"] == kernel and r["memoria"] == mem]
                if not sub:
                    continue
                ck = first_crossing(sub, "speedup_kernel")
                ct = first_crossing(sub, "speedup_total")
                fmt = lambda v: f"n >= {v}" if v else "no cruza en el rango medido"
                print(f"  {kernel:<11} {mem:<9} solo kernel: {fmt(ck):<32} con copias: {fmt(ct)}")


def main():
    if len(sys.argv) < 2:
        print(__doc__)
        sys.exit(2)
    csv_path = sys.argv[1]
    out_dir = sys.argv[2] if len(sys.argv) > 2 else (os.path.dirname(csv_path) or ".")
    os.makedirs(out_dir, exist_ok=True)
    rows = read_rows(csv_path)
    if not rows:
        print("El CSV no tiene filas")
        sys.exit(1)
    note = source_note(rows)
    print(f"{len(rows)} filas leídas de {csv_path} · {note}")
    plot_block_sweep(rows, os.path.join(out_dir, "tiempo_vs_bloque.png"), note)
    plot_threshold(rows, os.path.join(out_dir, "speedup_vs_n.png"), note)
    print_summary(rows)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile bench_numba.py
"""bench_numba.py — Espejo en Numba CUDA de los kernels de bench_gpu.cu.

Sin GPU (verificación en el simulador):  NUMBA_ENABLE_CUDASIM=1 python3 bench_numba.py --n 1000
Con GPU (Colab/Kaggle):                  python3 bench_numba.py --n 16777216

Comprueba que vector_add y saxpy dan el mismo resultado que NumPy para cada tamaño de bloque
(32, 64, ..., 1024) y, si hay GPU real, mide el tiempo del kernel con eventos (cuda.event), que son
el equivalente de cudaEvent_t: 1 calentamiento + 5 repeticiones, mediana.
Termina con código 0 si toda verificación pasa y 1 si alguna falla.
"""
import argparse
import os
import sys
import time

import numpy as np
from numba import cuda

BLOCK_SIZES = (32, 64, 128, 256, 512, 1024)
N_REPS = 5
ALPHA = 2.0
SIMULATOR = os.environ.get("NUMBA_ENABLE_CUDASIM") == "1"


@cuda.jit
def vector_add_kernel(a, b, c, n):
    i = cuda.grid(1)  # equivale a blockIdx.x * blockDim.x + threadIdx.x
    if i < n:
        c[i] = a[i] + b[i]


@cuda.jit
def saxpy_kernel(alpha, x, y, out, n):
    i = cuda.grid(1)
    if i < n:
        out[i] = alpha * x[i] + y[i]


def median(values):
    return sorted(values)[len(values) // 2]


def time_kernel_ms(launch):
    """Mediana en ms de N_REPS lanzamientos medidos con eventos de la GPU (tras 1 calentamiento)."""
    launch()
    cuda.synchronize()
    times = []
    for _ in range(N_REPS):
        start, stop = cuda.event(), cuda.event()
        start.record()
        launch()
        stop.record()
        stop.synchronize()  # igual que cudaEventSynchronize
        times.append(cuda.event_elapsed_time(start, stop))
    return median(times)


def main():
    parser = argparse.ArgumentParser(description="Espejo Numba de bench_gpu.cu")
    parser.add_argument("--n", type=int, default=1000 if SIMULATOR else 1 << 24,
                        help="elementos por vector (en el simulador conviene n <= 2000)")
    args = parser.parse_args()
    n = args.n

    a = (np.arange(n, dtype=np.float32) % 1000) * np.float32(0.001)
    b = np.float32(1.0) + (np.arange(n, dtype=np.float32) % 7)
    ref = {"vector_add": a + b, "saxpy": np.float32(ALPHA) * a + b}

    d_a, d_b = cuda.to_device(a), cuda.to_device(b)
    d_c = cuda.device_array_like(a)

    mode = "simulador (CUDASIM, sin tiempos)" if SIMULATOR else "GPU real"
    print(f"bench_numba: n = {n}, modo = {mode}")
    print(f"{'kernel':<11}{'bloque':>7}{'malla':>9}{'error máx':>12}{'estado':>8}{'kernel ms':>12}")
    all_ok = True
    t0 = time.perf_counter()
    for block in BLOCK_SIZES:
        grid = (n + block - 1) // block
        for name in ("vector_add", "saxpy"):
            if name == "vector_add":
                def launch(): vector_add_kernel[grid, block](d_a, d_b, d_c, n)
            else:
                def launch(): saxpy_kernel[grid, block](ALPHA, d_a, d_b, d_c, n)
            d_c.copy_to_device(np.zeros(n, dtype=np.float32))  # borra el resultado anterior
            launch()
            cuda.synchronize()
            err = float(np.max(np.abs(d_c.copy_to_host() - ref[name])))
            ok = err < 1e-5
            all_ok &= ok
            ms = "" if SIMULATOR else f"{time_kernel_ms(launch):12.4f}"
            print(f"{name:<11}{block:>7}{grid:>9}{err:>12.2e}{'OK' if ok else 'FALLA':>8}{ms}")
    print(f"Verificación contra NumPy: {'OK' if all_ok else 'FALLA'} "
          f"({time.perf_counter() - t0:.1f} s de pared)")
    sys.exit(0 if all_ok else 1)


if __name__ == "__main__":
    main()

## 3. Compilar (con optimización; nunca se mide con -O0)

In [ ]:
!nvcc -O2 -arch=sm_75 -o medir_eventos medir_eventos.cu
!nvcc -O2 -arch=sm_75 -o bench_gpu bench_gpu.cu
!nvcc -O2 -arch=sm_75 -o pinned_streams pinned_streams.cu
!ls -l medir_eventos bench_gpu pinned_streams

## 4. Ejecutar · ejemplo resuelto: cuatro formas de medir el mismo kernel

Antes de correr: ¿cuál de las cuatro filas dará el número más pequeño y por qué está mal?

In [ ]:
!./medir_eventos

## 5. Ejecutar · barrido de tamaño de bloque (experimento `bloque`)

Con tu predicción escrita (mejor bloque, peor bloque, razón), corre el barrido con `n = 2^24` y después con
`n = 4096`. Todas las corridas de esta sección agregan filas a `resultados.csv`.

In [ ]:
!rm -f resultados.csv
!./bench_gpu bloque 16777216 resultados.csv

In [ ]:
!./bench_gpu bloque 4096 resultados.csv

## 6. Ejecutar · umbral de tamaño: tabla con y sin transferencias (experimento `umbral`)

Predicción: ¿a partir de qué `n` gana la GPU contando solo el kernel? ¿Y contando las copias?

In [ ]:
!./bench_gpu umbral 256 resultados.csv

## 7. Investigar · perfilado con `nsys` (y `ncu` si está disponible)

`nsys profile --stats=true` imprime varias tablas. Mira estas tres y, en cada una, las columnas
**Time (%)**, **Total Time**, **Instances/Count**, **Avg** y **Name/Operation**:

- `cuda_api_sum`: tiempo de la CPU en cada llamada a la API (`cudaMemcpy`, `cudaMalloc`, `cudaEventSynchronize`).
- `cuda_gpu_kern_sum`: tiempo en la GPU de cada kernel (`vector_add_kernel`, `saxpy_kernel`).
- `cuda_gpu_mem_time_sum`: tiempo de las copias `[CUDA memcpy Host-to-Device]` y `[CUDA memcpy Device-to-Host]`.

Si `nsys` no existe en tu entorno, la celda lo dice y te quedas con los eventos, que miden lo mismo con menos detalle.

In [ ]:
import shutil
if shutil.which("nsys"):
    !nsys profile --stats=true -o perfil_s04 --force-overwrite true ./bench_gpu bloque 16777216 perfil.csv 2>&1 | grep -v "^$" | head -120
else:
    print("nsys no disponible: usa las columnas H2D/kernel/D2H que ya imprime bench_gpu (medidas con eventos)")

In [ ]:
# ncu (Nsight Compute) suele estar bloqueado en Colab (ERR_NVGPUCTRPERM). Si corre, mira la sección
# "GPU Speed Of Light": % del ancho de banda de memoria y % de cómputo que alcanza el kernel.
import shutil
if shutil.which("ncu"):
    !ncu --section SpeedOfLight --kernel-name regex:saxpy --launch-skip 6 --launch-count 1 ./bench_gpu bloque 16777216 perfil_ncu.csv 2>&1 | tail -40
else:
    print("ncu no disponible en este entorno")

## 8. Modificar · memoria *pinned* y *streams*

1. Repite el experimento `umbral` con memoria fija (`pinned`). Predice primero: ¿cambia el tiempo del kernel?
   ¿cambian las copias? ¿aparece el cruce "con copias" que antes no aparecía?
2. Corre `pinned_streams`: tres versiones del mismo trabajo (paginable, pinned, pinned + 4 streams).

In [ ]:
!./bench_gpu umbral 256 resultados.csv pinned

In [ ]:
!./pinned_streams

In [ ]:
# Opcional: ¿y con 2 u 8 streams? Predice antes si mejora, empeora o da igual.
!./pinned_streams 16777216 8

## 9. Graficar: tiempo vs. tamaño de bloque y speedup vs. n

In [ ]:
!python3 graficar.py resultados.csv .
from IPython.display import Image, display
display(Image("tiempo_vs_bloque.png"))
display(Image("speedup_vs_n.png"))

## 10. Opcional · el mismo barrido en Numba CUDA

Los kernels en Numba son el espejo de los de `bench_gpu.cu`; con GPU real también mide con eventos.

In [ ]:
!python3 bench_numba.py --n 16777216

## 11. Crear · tu propio kernel de la Práctica 6 con este protocolo

Copia tu kernel de la Práctica 6 a una celda `%%writefile`, compílalo y mídelo con el mismo esquema que `bench_gpu.cu`:
cuatro eventos (H2D, kernel, D2H), 1 calentamiento + 5 repeticiones, mediana, verificación contra la CPU y una
fila por configuración en un CSV con el mismo encabezado. Criterio de "listo": tabla con/sin copias para tres `n`,
barrido de bloque para un `n`, y una frase que diga si tu kernel está limitado por memoria o por cómputo (y por qué).

## 12. Guardar los resultados

Guarda el CSV con el nombre de la GPU y la fecha. En Colab se descarga; en Kaggle queda en `/kaggle/working`.
Súbelo a `resultados/` de tu repositorio de P6 junto con las dos PNG.

In [ ]:
import datetime, shutil, subprocess, os
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True,
                     text=True).stdout.strip().replace(" ", "_") or "sin_gpu"
stamp = datetime.date.today().isoformat()
nombre = f"resultados_S04_{gpu}_{stamp}.csv"
shutil.copy("resultados.csv", nombre)
for png in ("tiempo_vs_bloque.png", "speedup_vs_n.png"):
    if os.path.exists(png):
        shutil.copy(png, png.replace(".png", f"_{gpu}_{stamp}.png"))
print("Guardado:", nombre)
try:
    from google.colab import files  # solo existe en Colab
    files.download(nombre)
except ImportError:
    print("Fuera de Colab: el archivo queda en el directorio de trabajo")